In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common

cfg = common.load_config("dev")
cat = cfg["catalog"]
want = {"hospitals": 5, "departments": 30, "doctors": 73, "patients": 654, "appointments": 2191,
        "admissions": 269, "treatments": 506, "pharmacy_sales": 1644, "invoices": 2168,
        "payments": 1975, "insurance_claims": 81}
got = {t: spark.table(f"{cat}.silver.{t}").count() for t in want}
q = spark.table(f"{cat}.silver.quarantine").filter("status = 'open'")
count = lambda rule: {r["table_name"]: r["count"] for r in q.filter(f"rule = '{rule}'").groupBy("table_name").count().collect()}
orphans, future = count("orphan"), count("future_date")
unknown = spark.table(f"{cat}.silver.appointments").filter("orphan_of IS NOT NULL").count()
display(q.select("table_name", "row_key", "rule", "detail", "status").orderBy("rule", "table_name", "row_key"))
print(got)
print("orphans waiting", orphans, "| future dates", future, "| accepted with unknown doctor", unknown)
ok = (got == want and orphans == {"appointments": 1, "invoices": 1, "payments": 1}
      and future == {"invoices": 5, "payments": 1} and unknown == 4)
print("PASS" if ok else "FAIL")